# Deploy Synthefy-Nori from AWS Marketplace

This notebook deploys the subscribed Synthefy-Nori model package in your AWS account and invokes it with the released `SynthefyNoriClient`. One SageMaker model and endpoint serve all three peer models:

- `nori-6m`
- `nori-30m`
- `nori-30m-thinking-medium`

Each request selects a model explicitly. There is no default or primary model. The notebook also runs a batch transform job, shows the equivalent raw streaming request, and removes the billable resources at the end.

> SageMaker GPU resources incur AWS infrastructure charges while they exist. Run the cleanup section even if an earlier cell fails.

## 1. Prerequisites

Before running the notebook:

1. Subscribe to **Synthefy-Nori** in AWS Marketplace and choose **Launch your software**.
2. Copy the model package ARN for the AWS Region where you will deploy it.
3. Have a SageMaker execution-role ARN that SageMaker can assume.
4. Confirm that the Region has quota for one `ml.g5.xlarge` endpoint and batch transform instance.
5. Provide an existing S3 bucket in the same Region for the batch example.

The identity running this notebook needs permission to create and delete SageMaker models, endpoint configurations, endpoints, and transform jobs; pass the execution role; and read and write the selected S3 prefix.

## 2. Install the client

In [ ]:
%pip install -q "synthefy==6.3.0" boto3

## 3. Configure the deployment

Replace the three placeholder values below. Use the model package ARN shown by AWS Marketplace after subscribing, not the seller's internal validation ARN. Resource names include a random suffix so repeated runs do not collide.

In [ ]:
import json
import time
import uuid

import boto3
from botocore.exceptions import ClientError
from synthefy import SynthefyNoriClient

REGION = boto3.Session().region_name or "us-east-1"
MODEL_PACKAGE_ARN = "PASTE_SUBSCRIBED_MODEL_PACKAGE_ARN"
EXECUTION_ROLE_ARN = "PASTE_SAGEMAKER_EXECUTION_ROLE_ARN"
S3_BUCKET = "PASTE_EXISTING_BUCKET_NAME"
INSTANCE_TYPE = "ml.g5.xlarge"

assert MODEL_PACKAGE_ARN.startswith("arn:aws:sagemaker:"), "Set MODEL_PACKAGE_ARN"
assert EXECUTION_ROLE_ARN.startswith("arn:aws:iam::"), "Set EXECUTION_ROLE_ARN"
assert not S3_BUCKET.startswith("PASTE_"), "Set S3_BUCKET"

run_id = uuid.uuid4().hex[:8]
resource_prefix = f"nori-marketplace-{run_id}"
model_name = f"{resource_prefix}-model"
endpoint_config_name = f"{resource_prefix}-config"
endpoint_name = f"{resource_prefix}-endpoint"
transform_job_name = f"{resource_prefix}-batch"
s3_prefix = f"nori-marketplace-notebook/{resource_prefix}"

sm = boto3.client("sagemaker", region_name=REGION)
s3 = boto3.client("s3", region_name=REGION)
runtime = boto3.client("sagemaker-runtime", region_name=REGION)

tags = [
    {"Key": "Project", "Value": "synthefy-nori-marketplace-notebook"},
    {"Key": "RunId", "Value": run_id},
]

print(f"Region: {REGION}")
print(f"Resource prefix: {resource_prefix}")

## 4. Prepare a sample request

Nori performs in-context regression. `X_train` and `y_train` are labeled context supplied with the request; `X_test` contains the rows to predict. The model returns one prediction per `X_test` row.

In [ ]:
X_train = [[0.0, 1.0], [1.0, 0.0], [0.5, 0.5], [0.2, 0.8]]
y_train = [0.1, 0.9, 0.5, 0.3]
X_test = [[0.3, 0.7], [0.8, 0.2]]

sample_payload = {
    "model": "nori-6m",
    "task": "regression",
    "X_train": X_train,
    "y_train": y_train,
    "X_test": X_test,
}

## 5. Create one SageMaker model

The Marketplace model package contains the serving image and model artifacts. Do not add an inference-specification selector or create one SageMaker model per Nori variant.

In [ ]:
sm.create_model(
    ModelName=model_name,
    ExecutionRoleArn=EXECUTION_ROLE_ARN,
    PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
    Tags=tags,
)
print(f"Created SageMaker model: {model_name}")

## 6. Run batch transform

This runs before endpoint creation so the notebook needs quota for only one GPU instance at a time. The transform job stops its instance automatically when it completes.

In [ ]:
batch_input_key = f"{s3_prefix}/batch-input/sample.json"
batch_output_prefix = f"{s3_prefix}/batch-output"
batch_input_uri = f"s3://{S3_BUCKET}/{batch_input_key}"
batch_output_uri = f"s3://{S3_BUCKET}/{batch_output_prefix}/"

s3.put_object(
    Bucket=S3_BUCKET,
    Key=batch_input_key,
    Body=json.dumps(sample_payload, separators=(",", ":")).encode(),
    ContentType="application/json",
)

sm.create_transform_job(
    TransformJobName=transform_job_name,
    ModelName=model_name,
    TransformInput={
        "DataSource": {
            "S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": batch_input_uri}
        },
        "ContentType": "application/json",
        "SplitType": "None",
    },
    TransformOutput={"S3OutputPath": batch_output_uri, "Accept": "application/json"},
    TransformResources={"InstanceType": INSTANCE_TYPE, "InstanceCount": 1},
    Tags=tags,
)

while True:
    description = sm.describe_transform_job(TransformJobName=transform_job_name)
    status = description["TransformJobStatus"]
    print(f"TransformJobStatus={status}")
    if status == "Completed":
        break
    if status in {"Failed", "Stopped"}:
        raise RuntimeError(description.get("FailureReason", f"Transform job {status}"))
    time.sleep(30)

batch_output_key = f"{batch_output_prefix}/sample.json.out"
batch_body = s3.get_object(Bucket=S3_BUCKET, Key=batch_output_key)["Body"].read()
batch_result = json.loads(batch_body)
print(json.dumps(batch_result, indent=2))

## 7. Create a real-time endpoint

Endpoint creation commonly takes 15 to 30 minutes and can take longer when the selected instance type has limited regional capacity.

In [ ]:
sm.create_endpoint_config(
    EndpointConfigName=endpoint_config_name,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "InitialInstanceCount": 1,
            "InstanceType": INSTANCE_TYPE,
            "InitialVariantWeight": 1.0,
            "ContainerStartupHealthCheckTimeoutInSeconds": 1800,
        }
    ],
    Tags=tags,
)
sm.create_endpoint(
    EndpointName=endpoint_name,
    EndpointConfigName=endpoint_config_name,
    Tags=tags,
)

while True:
    description = sm.describe_endpoint(EndpointName=endpoint_name)
    status = description["EndpointStatus"]
    print(f"EndpointStatus={status}")
    if status == "InService":
        break
    if status in {"Failed", "OutOfService"}:
        raise RuntimeError(description.get("FailureReason", f"Endpoint {status}"))
    time.sleep(30)

print(f"Endpoint ready: {endpoint_name}")

## 8. Invoke all three models with SynthefyNoriClient

The client uses the standard AWS credential chain, signs the request, sets the streaming attribute, joins heartbeat chunks, and returns the prediction array.

In [ ]:
models = ["nori-6m", "nori-30m", "nori-30m-thinking-medium"]
predictions_by_model = {}

for model in models:
    with SynthefyNoriClient(
        mode="sagemaker",
        model=model,
        endpoint_name=endpoint_name,
        region_name=REGION,
        timeout=480,
    ) as client:
        predictions = client.predict(X_train, y_train, X_test)

    assert len(predictions) == len(X_test)
    predictions_by_model[model] = predictions
    print(f"{model}: {predictions}")

## 9. Optional: invoke the streaming API directly

Use this form when integrating without the Synthefy client. Every real-time model uses `InvokeEndpointWithResponseStream`, including `nori-6m`. Heartbeat payload parts contain JSON whitespace; join all payload bytes before decoding the result.

In [ ]:
response = runtime.invoke_endpoint_with_response_stream(
    EndpointName=endpoint_name,
    ContentType="application/json",
    Accept="application/json",
    CustomAttributes="synthefy-response-stream=v1",
    Body=json.dumps(sample_payload, separators=(",", ":")).encode(),
)

body = b"".join(
    event["PayloadPart"]["Bytes"]
    for event in response["Body"]
    if "PayloadPart" in event
)
raw_result = json.loads(body)
if "error" in raw_result:
    raise RuntimeError(raw_result["error"])
print(json.dumps(raw_result, indent=2))

## 10. Cleanup

Run this cell even if deployment or invocation failed. It deletes the endpoint, stops an interrupted transform job if necessary, and then removes the endpoint configuration, model, and notebook-owned S3 objects. The completed transform-job record remains visible in SageMaker but has no running instance.

In [ ]:
def is_missing(error):
    return error.response.get("Error", {}).get("Code") == "ValidationException"


try:
    sm.delete_endpoint(EndpointName=endpoint_name)
    print(f"Deleting endpoint: {endpoint_name}")
    sm.get_waiter("endpoint_deleted").wait(
        EndpointName=endpoint_name,
        WaiterConfig={"Delay": 30, "MaxAttempts": 80},
    )
except ClientError as error:
    if not is_missing(error):
        raise

try:
    description = sm.describe_transform_job(TransformJobName=transform_job_name)
    status = description["TransformJobStatus"]
    if status == "InProgress":
        sm.stop_transform_job(TransformJobName=transform_job_name)
        print(f"Stopping transform job: {transform_job_name}")
    while status in {"InProgress", "Stopping"}:
        time.sleep(15)
        status = sm.describe_transform_job(
            TransformJobName=transform_job_name
        )["TransformJobStatus"]
except ClientError as error:
    if not is_missing(error):
        raise

try:
    sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    print(f"Deleted endpoint config: {endpoint_config_name}")
except ClientError as error:
    if not is_missing(error):
        raise

try:
    sm.delete_model(ModelName=model_name)
    print(f"Deleted model: {model_name}")
except ClientError as error:
    if not is_missing(error):
        raise

paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=s3_prefix):
    objects = [{"Key": item["Key"]} for item in page.get("Contents", [])]
    if objects:
        s3.delete_objects(Bucket=S3_BUCKET, Delete={"Objects": objects, "Quiet": True})
print(f"Deleted notebook S3 objects under s3://{S3_BUCKET}/{s3_prefix}/")